# Phase 7: Recursive Self-Invocation via PAUSE/RESUME

**Architecture:** The model learns the Karatsuba algorithm structure — it decides
WHEN to recurse (emitting [PAUSE]) and HOW to combine results (after [RESUME]).
An external runtime manages the call stack; the model drives the algorithm.

**Trace format:** Single-level traces (~114 tokens for 8-bit vs ~371 for full DFS).
Sub-problems are replaced with [PAUSE]/[RESUME] boundaries.

**Runtime:** A100 GPU. Runtime → Change runtime type → A100.

In [ ]:
# Cell 1: Setup
import os
if not os.path.exists('/content/recursive-transformers/src'):
    !git clone https://github.com/dhruvsyam123/recursive-transformers.git /content/recursive-transformers
else:
    !cd /content/recursive-transformers && git pull
%cd /content/recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops

import jax
print(f'JAX devices: {jax.devices()}')
print(f'JAX backend: {jax.default_backend()}')

In [ ]:
# Cell 2: Model — same as Phase 5 but with PAUSE/RESUME tokens
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import math
import numpy as np

from src.model.looped_transformer import (
    MultiHeadSelfAttention, FeedForward, RMSNorm, count_parameters,
)
from src.data.karatsuba_trace import int_to_bits, bits_to_int

# --- Extended vocabulary (inline, no source file changes) ---
# Original token IDs from src/data/tokenizer.py
PAD_ID = 0
SEP_ID = 1
INPUT_ID = 2
SPLIT_ID = 3
SUB_MUL_0_ID = 4
SUB_MUL_1_ID = 5
SUB_MUL_2_ID = 6
ADD_ID = 7
SUB_ID = 8
COMBINE_ID = 9
OUTPUT_ID = 10
BASE_MUL_ID = 11
BIT_0_ID = 12
BIT_1_ID = 13
# New tokens (appended at end of vocab to preserve all existing IDs)
PAUSE_ID = 143
RESUME_ID = 144
VOCAB_SIZE = 145

# Step types for position encoding
STEP_INPUT = 0
STEP_SPLIT = 1
STEP_SUB_MUL_0 = 2
STEP_SUB_MUL_1 = 3
STEP_SUB_MUL_2 = 4
STEP_ADD = 5
STEP_SUB = 6
STEP_COMBINE = 7
STEP_OUTPUT = 8
STEP_BASE_MUL = 9
STEP_PAUSE = 10
STEP_RESUME = 11

TAG_TO_STEP = {
    INPUT_ID: STEP_INPUT, SPLIT_ID: STEP_SPLIT,
    SUB_MUL_0_ID: STEP_SUB_MUL_0, SUB_MUL_1_ID: STEP_SUB_MUL_1,
    SUB_MUL_2_ID: STEP_SUB_MUL_2, ADD_ID: STEP_ADD,
    SUB_ID: STEP_SUB, COMBINE_ID: STEP_COMBINE,
    OUTPUT_ID: STEP_OUTPUT, BASE_MUL_ID: STEP_BASE_MUL,
    PAUSE_ID: STEP_PAUSE, RESUME_ID: STEP_RESUME,
}

ALL_TAG_IDS = set(TAG_TO_STEP.keys())

def bit_to_token(b):
    return BIT_0_ID if b == 0 else BIT_1_ID

def token_to_bit(t):
    return 0 if t == BIT_0_ID else 1


# --- Sinusoidal encoding (same as Phase 5) ---
def sinusoidal_encoding(positions, d_model):
    pos = positions.astype(jnp.float32)
    if pos.ndim == 0:
        pos = pos[None]
        squeeze = True
    else:
        squeeze = False
    dim_indices = jnp.arange(0, d_model, 2, dtype=jnp.float32)
    freqs = jnp.exp(-dim_indices * (math.log(10000.0) / d_model))
    angles = pos[:, None] * freqs[None, :]
    enc = jnp.zeros((pos.shape[0], d_model))
    enc = enc.at[:, 0::2].set(jnp.sin(angles))
    enc = enc.at[:, 1::2].set(jnp.cos(angles))
    if squeeze:
        return enc[0]
    return enc


class AllSinusoidalHierarchicalPos(eqx.Module):
    d_model: int = eqx.field(static=True)
    comp_dim: int = eqx.field(static=True)
    def __init__(self, d_model):
        assert d_model % 4 == 0
        self.d_model = d_model
        self.comp_dim = d_model // 4
    def __call__(self, bit_sig, depth, sub_id, step_type):
        return jnp.concatenate([
            sinusoidal_encoding(bit_sig, self.comp_dim),
            sinusoidal_encoding(depth, self.comp_dim),
            sinusoidal_encoding(sub_id, self.comp_dim),
            sinusoidal_encoding(step_type, self.comp_dim),
        ], axis=-1)


class LoopBlock(eqx.Module):
    attention: MultiHeadSelfAttention
    ffn: FeedForward
    norm1: RMSNorm
    norm2: RMSNorm
    d_model: int = eqx.field(static=True)
    def __init__(self, d_model, n_heads, d_ff, *, key):
        k1, k2 = jax.random.split(key)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, key=k1)
        self.ffn = FeedForward(d_model, d_ff, key=k2)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.d_model = d_model
    def __call__(self, x, timestep, mask=None):
        t_emb = sinusoidal_encoding(timestep, self.d_model)
        x_cond = x + t_emb[None, :]
        normed = jax.vmap(self.norm1)(x_cond)
        x = x + self.attention(normed, mask=mask)
        normed = jax.vmap(self.norm2)(x)
        x = x + jax.vmap(self.ffn)(normed)
        return x


class KaratsubaModel(eqx.Module):
    token_embed: eqx.nn.Embedding
    hier_pos: AllSinusoidalHierarchicalPos
    block: LoopBlock
    final_norm: RMSNorm
    output_head: eqx.nn.Linear
    inject_scale: jnp.ndarray
    def __call__(self, tokens, n_loops, bit_sig, depth, sub_id, step_type):
        x0 = jax.vmap(self.token_embed)(tokens)
        pos = self.hier_pos(bit_sig, depth, sub_id, step_type)
        x0 = x0 + pos
        x = x0
        seq_len = tokens.shape[0]
        mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        def scan_fn(hidden, timestep):
            hidden = hidden + self.inject_scale * x0
            hidden = self.block(hidden, timestep, mask=mask)
            return hidden, None
        x, _ = jax.lax.scan(scan_fn, x, jnp.arange(n_loops))
        x = jax.vmap(self.final_norm)(x)
        return jax.vmap(self.output_head)(x)


D_MODEL = 256
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)

model = KaratsubaModel(
    token_embed=eqx.nn.Embedding(VOCAB_SIZE, D_MODEL, key=k1),
    hier_pos=AllSinusoidalHierarchicalPos(D_MODEL),
    block=LoopBlock(D_MODEL, n_heads=8, d_ff=512, key=k2),
    final_norm=RMSNorm(D_MODEL),
    output_head=eqx.nn.Linear(D_MODEL, VOCAB_SIZE, key=k3),
    inject_scale=jnp.array(0.1),
)
print(f'Parameters: {count_parameters(model):,}')

In [ ]:
# Cell 3: PAUSE/RESUME trace generator + dataset + mask

# --- Single-level trace generator ---
def generate_single_level_trace(x, y, n_bits, base_case_bits=4):
    """Generate a single-level Karatsuba trace with PAUSE/RESUME.
    Returns list of (token_id, position_dict) tuples.
    
    Base case (n_bits <= base_case_bits): [INPUT] bits [BASE_MUL] product [OUTPUT] product
    Recursive case: [INPUT] bits [SPLIT] ... [SUB_MUL_0] operands [PAUSE] [RESUME] result ... [OUTPUT] result
    """
    assert x >= 0 and y >= 0
    assert x < (1 << n_bits) and y < (1 << n_bits)
    product = x * y
    product_bits = 2 * n_bits
    
    seq = []  # list of (token_id, {bit_significance, step_type})
    
    def add_tag(tag_id, step_type):
        seq.append((tag_id, {'bit_significance': 200 + step_type, 'step_type': step_type}))
    
    def add_bits(bits_list, step_type, bit_offset=0):
        for i, b in enumerate(bits_list):
            seq.append((bit_to_token(b), {'bit_significance': bit_offset + i, 'step_type': step_type}))
    
    # --- Base case ---
    if n_bits <= base_case_bits:
        add_tag(INPUT_ID, STEP_INPUT)
        add_bits(int_to_bits(x, n_bits), STEP_INPUT)
        add_bits(int_to_bits(y, n_bits), STEP_INPUT)
        add_tag(BASE_MUL_ID, STEP_BASE_MUL)
        add_bits(int_to_bits(product, product_bits), STEP_BASE_MUL)
        add_tag(OUTPUT_ID, STEP_OUTPUT)
        add_bits(int_to_bits(product, product_bits), STEP_OUTPUT)
        return seq, product
    
    # --- Recursive case ---
    half = n_bits // 2
    hi_bits = n_bits - half
    mask_low = (1 << half) - 1
    
    x_lo, x_hi = x & mask_low, x >> half
    y_lo, y_hi = y & mask_low, y >> half
    
    # Compute sub-products classically (for ground truth in RESUME)
    z0 = x_lo * y_lo
    z2 = x_hi * y_hi
    sum_x = x_lo + x_hi
    sum_y = y_lo + y_hi
    z1_raw = sum_x * sum_y
    z1 = z1_raw - z0 - z2
    result = z0 + (z1 << half) + (z2 << (2 * half))
    assert result == product, f'{x}*{y}: {result} != {product}'
    
    # Sub-problem bit widths
    z0_n = half
    z2_n = hi_bits
    sum_actual_bits = max(half, hi_bits) + 1
    z1_n = sum_actual_bits
    
    # [INPUT]
    add_tag(INPUT_ID, STEP_INPUT)
    add_bits(int_to_bits(x, n_bits), STEP_INPUT)
    add_bits(int_to_bits(y, n_bits), STEP_INPUT)
    
    # [SPLIT]
    add_tag(SPLIT_ID, STEP_SPLIT)
    add_bits(int_to_bits(x_hi, hi_bits), STEP_SPLIT)
    add_bits(int_to_bits(x_lo, half), STEP_SPLIT)
    add_bits(int_to_bits(y_hi, hi_bits), STEP_SPLIT)
    add_bits(int_to_bits(y_lo, half), STEP_SPLIT)
    
    # [SUB_MUL_0] operands [PAUSE] / [RESUME] z0
    add_tag(SUB_MUL_0_ID, STEP_SUB_MUL_0)
    add_bits(int_to_bits(x_lo, z0_n), STEP_SUB_MUL_0)
    add_bits(int_to_bits(y_lo, z0_n), STEP_SUB_MUL_0)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z0, 2 * z0_n), STEP_RESUME)
    
    # [SUB_MUL_2] operands [PAUSE] / [RESUME] z2
    add_tag(SUB_MUL_2_ID, STEP_SUB_MUL_2)
    add_bits(int_to_bits(x_hi, z2_n), STEP_SUB_MUL_2)
    add_bits(int_to_bits(y_hi, z2_n), STEP_SUB_MUL_2)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z2, 2 * z2_n), STEP_RESUME)
    
    # [ADD] sums
    add_tag(ADD_ID, STEP_ADD)
    add_bits(int_to_bits(sum_x, sum_actual_bits), STEP_ADD)
    add_bits(int_to_bits(sum_y, sum_actual_bits), STEP_ADD)
    
    # [SUB_MUL_1] operands [PAUSE] / [RESUME] z1_raw
    add_tag(SUB_MUL_1_ID, STEP_SUB_MUL_1)
    add_bits(int_to_bits(sum_x, z1_n), STEP_SUB_MUL_1)
    add_bits(int_to_bits(sum_y, z1_n), STEP_SUB_MUL_1)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z1_raw, 2 * z1_n), STEP_RESUME)
    
    # [SUB] z1 = z1_raw - z0 - z2
    add_tag(SUB_ID, STEP_SUB)
    add_bits(int_to_bits(z1, product_bits), STEP_SUB)
    
    # [COMBINE] result
    add_tag(COMBINE_ID, STEP_COMBINE)
    add_bits(int_to_bits(result, product_bits), STEP_COMBINE)
    
    # [OUTPUT] result
    add_tag(OUTPUT_ID, STEP_OUTPUT)
    add_bits(int_to_bits(result, product_bits), STEP_OUTPUT)
    
    return seq, product


# --- Test trace generator ---
print('=== Trace Generator Test ===')
for x, y, nb in [(7, 5, 4), (179, 214, 8), (15, 15, 4), (0, 200, 8)]:
    seq, prod = generate_single_level_trace(x, y, nb)
    expected = x * y
    status = 'OK' if prod == expected else 'FAIL'
    tag_names = {INPUT_ID:'IN', SPLIT_ID:'SP', SUB_MUL_0_ID:'M0', SUB_MUL_1_ID:'M1',
                 SUB_MUL_2_ID:'M2', ADD_ID:'AD', SUB_ID:'SU', COMBINE_ID:'CO',
                 OUTPUT_ID:'OU', BASE_MUL_ID:'BM', PAUSE_ID:'PA', RESUME_ID:'RE'}
    tags = [tag_names.get(t, '.') for t, _ in seq if t in ALL_TAG_IDS]
    print(f'{x:3d} * {y:3d} ({nb}-bit) = {prod} [{status}] {len(seq)} tokens  tags: {"-".join(tags)}')


# --- Dataset creation ---
def make_dataset(bit_widths_and_counts, base_case_bits=4, seed=42):
    """Generate training examples as padded numpy arrays."""
    rng = np.random.RandomState(seed)
    examples = []  # list of (token_ids, positions, product)
    
    for n_bits, n_samples in bit_widths_and_counts:
        max_val = 1 << n_bits
        if n_bits <= 4:  # exhaustive
            pairs = [(x, y) for x in range(max_val) for y in range(max_val)]
        else:
            pairs = [(rng.randint(0, max_val), rng.randint(0, max_val)) for _ in range(n_samples)]
        
        for x, y in pairs:
            seq, prod = generate_single_level_trace(x, y, n_bits, base_case_bits)
            token_ids = np.array([t for t, _ in seq], dtype=np.int32)
            # Positions: just bit_significance and step_type (depth=0, sub_id=0 always)
            bit_sigs = np.array([p['bit_significance'] for _, p in seq], dtype=np.int32)
            step_types = np.array([p['step_type'] for _, p in seq], dtype=np.int32)
            examples.append((token_ids, bit_sigs, step_types, x, y, n_bits, prod))
    
    return examples


def get_batches(examples, batch_size, max_len, shuffle=True, rng_seed=None):
    """Yield padded batches."""
    rng = np.random.RandomState(rng_seed)
    indices = np.arange(len(examples))
    if shuffle:
        rng.shuffle(indices)
    
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start+batch_size]
        bs = len(batch_idx)
        
        tk = np.zeros((bs, max_len), dtype=np.int32)  # PAD_ID = 0
        bs_arr = np.zeros((bs, max_len), dtype=np.int32)
        st_arr = np.zeros((bs, max_len), dtype=np.int32)
        pm = np.zeros((bs, max_len), dtype=np.float32)
        
        for i, idx in enumerate(batch_idx):
            token_ids, bit_sigs, step_types, x, y, nb, prod = examples[idx]
            sl = len(token_ids)
            tk[i, :sl] = token_ids
            bs_arr[i, :sl] = bit_sigs
            st_arr[i, :sl] = step_types
            pm[i, :sl] = 1.0
        
        yield {
            'token_ids': tk,
            'bit_sig': bs_arr,
            'step_type': st_arr,
            'pad_mask': pm,
        }


# --- Loss mask ---
def make_pause_resume_mask(token_ids_full, pad_mask_full):
    """Mask out INPUT bits and RESUME bits (unpredictable).
    Uses cumulative scan to track most-recent tag.
    Returns mask aligned with targets (shifted by 1)."""
    # Identify tag positions
    is_input_tag = (token_ids_full == INPUT_ID)
    is_resume_tag = (token_ids_full == RESUME_ID)
    is_bit = (token_ids_full == BIT_0_ID) | (token_ids_full == BIT_1_ID)
    
    # Track "after INPUT" and "after RESUME" regions
    # A region ends when any non-bit token (tag) appears
    is_any_tag = jnp.zeros_like(token_ids_full, dtype=jnp.bool_)
    for tid in [INPUT_ID, SPLIT_ID, SUB_MUL_0_ID, SUB_MUL_1_ID, SUB_MUL_2_ID,
                ADD_ID, SUB_ID, COMBINE_ID, OUTPUT_ID, BASE_MUL_ID, PAUSE_ID, RESUME_ID]:
        is_any_tag = is_any_tag | (token_ids_full == tid)
    
    # For each position, what was the last tag? Use cummax trick:
    # Assign each tag position an increasing index, forward-fill for bit positions
    # Then check if that tag was INPUT or RESUME
    #
    # Simpler approach: track cumulative sums of INPUT and RESUME tags,
    # and cumulative sums of ALL tags. A bit is "after INPUT/RESUME" if
    # the count of INPUT+RESUME tags > count of OTHER tags since the last tag.
    #
    # Even simpler: a bit is unpredictable if no non-(INPUT|RESUME) tag has
    # appeared since the last (INPUT|RESUME) tag.
    
    # Forward-fill the last tag ID
    # Use the fact that tag_ids are > 0 and bit tokens we set to 0
    tag_or_zero = jnp.where(is_any_tag, token_ids_full, 0)
    
    # Cumulative max of tag positions (assigns each position its last tag)
    # Since we want forward-fill, we use a scan
    def scan_fill(carry, x):
        val = jnp.where(x > 0, x, carry)
        return val, val
    
    # Vectorize across batch
    def forward_fill_row(row):
        _, filled = jax.lax.scan(scan_fill, jnp.int32(0), row)
        return filled
    
    last_tag = jax.vmap(forward_fill_row)(tag_or_zero)
    
    # A bit is unpredictable if its last_tag is INPUT or RESUME
    unpredictable = is_bit & ((last_tag == INPUT_ID) | (last_tag == RESUME_ID))
    predictable = ~unpredictable
    full_mask = pad_mask_full * predictable.astype(jnp.float32)
    
    # Shift by 1 to align with targets
    return full_mask[:, 1:]


# --- Build datasets ---
print('\n=== Building Datasets ===')
train_4bit = make_dataset([(4, 256)], base_case_bits=4, seed=42)
train_8bit = make_dataset([(8, 4000)], base_case_bits=4, seed=42)

ml4 = max(len(ex[0]) for ex in train_4bit) + 2
ml8 = max(len(ex[0]) for ex in train_8bit) + 2

print(f'4-bit: {len(train_4bit)} examples, max_len={ml4}')
print(f'8-bit: {len(train_8bit)} examples, max_len={ml8}')

# Verify mask on a sample
print('\n=== Mask Verification ===')
for batch in get_batches(train_8bit[:4], batch_size=4, max_len=ml8, shuffle=False):
    tk = jnp.array(batch['token_ids'])
    pm = jnp.array(batch['pad_mask'])
    mask = make_pause_resume_mask(tk, pm)
    
    # Show first example
    tag_names = {INPUT_ID:'[IN]', SPLIT_ID:'[SP]', SUB_MUL_0_ID:'[M0]', SUB_MUL_1_ID:'[M1]',
                 SUB_MUL_2_ID:'[M2]', ADD_ID:'[AD]', SUB_ID:'[SU]', COMBINE_ID:'[CO]',
                 OUTPUT_ID:'[OU]', BASE_MUL_ID:'[BM]', PAUSE_ID:'[PA]', RESUME_ID:'[RE]',
                 BIT_0_ID:'0', BIT_1_ID:'1', PAD_ID:'_'}
    tgt = tk[0, 1:]  # targets
    m = mask[0]
    print(f'{'Pos':>4} {'Input':>6} {'Target':>6} {'Mask':>5}')
    print('-' * 25)
    for i in range(min(30, int(pm[0].sum()) - 1)):
        inp_name = tag_names.get(int(tk[0, i]), '?')
        tgt_name = tag_names.get(int(tgt[i]), '?')
        print(f'{i:4d} {inp_name:>6} {tgt_name:>6} {float(m[i]):>5.0f}')
    break

In [ ]:
# Cell 4: Training
import random

schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=1e-3, warmup_steps=500,
    decay_steps=50000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.15)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def make_train_step(n_loops):
    @eqx.filter_jit
    def train_step(model, opt_state, token_ids, pad_mask, bit_sig, step_type):
        input_ids = token_ids[:, :-1]
        target_ids = token_ids[:, 1:]
        inp_bs = bit_sig[:, :-1]
        inp_st = step_type[:, :-1]
        mask = make_pause_resume_mask(token_ids, pad_mask)
        
        # Positions: depth=0, sub_id=0 always for single-level traces
        zeros = jnp.zeros_like(inp_bs)
        
        def loss_fn(model):
            def fwd(ids, bs, st):
                return model(ids, n_loops, bs, zeros[0], zeros[0], st)
            logits = jax.vmap(fwd)(input_ids, inp_bs, inp_st)
            log_probs = jax.nn.log_softmax(logits, axis=-1)
            targets_oh = jax.nn.one_hot(target_ids, VOCAB_SIZE)
            tok_loss = -jnp.sum(targets_oh * log_probs, axis=-1)
            return jnp.sum(tok_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)
        
        loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
        updates, new_opt = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
        return eqx.apply_updates(model, updates), new_opt, loss
    return train_step

train_steps = {n: make_train_step(n) for n in [4, 6, 8]}

# Teacher-forced eval
def teacher_forced_eval(model, examples, max_len, n_loops=8, n_samples=200):
    subset = examples[:n_samples]
    correct, total = 0, 0
    for batch in get_batches(subset, batch_size=16, max_len=max_len, shuffle=False):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['pad_mask'])
        bs = jnp.array(batch['bit_sig'])
        st = jnp.array(batch['step_type'])
        inp = tk[:, :-1]
        tgt = tk[:, 1:]
        inp_bs = bs[:, :-1]
        inp_st = st[:, :-1]
        zeros = jnp.zeros_like(inp_bs)
        em = make_pause_resume_mask(tk, pm)
        def fwd(ids, b, s):
            return model(ids, n_loops, b, zeros[0], zeros[0], s)
        logits = jax.vmap(fwd)(inp, inp_bs, inp_st)
        preds = jnp.argmax(logits, axis=-1)
        matches = (preds == tgt) | (em == 0)
        correct += int(jnp.all(matches, axis=-1).sum())
        total += tk.shape[0]
    return correct, total

rng = random.Random(42)
print('Training (PAUSE/RESUME traces, weight_decay=0.15)...')
for epoch in range(500):
    epoch_loss, n_batches = 0.0, 0
    
    # 4-bit base case batches
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_4bit, batch_size=64, max_len=ml4):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['pad_mask'])
        bs = jnp.array(batch['bit_sig'])
        st = jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss)
        n_batches += 1
    
    # 8-bit single-level batches
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_8bit, batch_size=16, max_len=ml8):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['pad_mask'])
        bs = jnp.array(batch['bit_sig'])
        st = jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss)
        n_batches += 1
    
    if epoch % 50 == 0:
        avg = epoch_loss / n_batches
        c8, t8 = teacher_forced_eval(model, train_8bit, ml8)
        print(f'Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit TF: {c8}/{t8} = {c8/t8:.1%}')

print('\nTraining complete.')
c8, t8 = teacher_forced_eval(model, train_8bit, ml8)
print(f'Final 8-bit teacher-forced: {c8}/{t8} = {c8/t8:.1%}')

In [ ]:
# Cell 5: Autoregressive recursive inference runtime

def model_predict_next(model, token_ids, bit_sigs, step_types, n_loops=8):
    """Run model on current sequence, return predicted next token."""
    tokens = jnp.array(token_ids, dtype=jnp.int32)
    bs = jnp.array(bit_sigs, dtype=jnp.int32)
    st = jnp.array(step_types, dtype=jnp.int32)
    zeros = jnp.zeros_like(bs)
    logits = model(tokens, n_loops, bs, zeros, zeros, st)
    return int(jnp.argmax(logits[-1]))


def extract_sub_problem(token_ids):
    """Extract sub-problem operands from tokens before [PAUSE].
    Looks backward from end to find [SUB_MUL_x] tag, extracts bits between it and end."""
    # Find the last SUB_MUL tag
    sub_mul_ids = {SUB_MUL_0_ID, SUB_MUL_1_ID, SUB_MUL_2_ID}
    last_sub_pos = -1
    for i in range(len(token_ids) - 1, -1, -1):
        if token_ids[i] in sub_mul_ids:
            last_sub_pos = i
            break
    
    if last_sub_pos == -1:
        return None
    
    # Extract bits between SUB_MUL tag and end (PAUSE was the last token, not yet appended)
    bits = []
    for i in range(last_sub_pos + 1, len(token_ids)):
        t = token_ids[i]
        if t == BIT_0_ID:
            bits.append(0)
        elif t == BIT_1_ID:
            bits.append(1)
        elif t in ALL_TAG_IDS:
            break  # hit another tag
    
    if len(bits) == 0 or len(bits) % 2 != 0:
        return None
    
    half = len(bits) // 2
    a_bits = bits[:half]
    b_bits = bits[half:]
    a = bits_to_int(a_bits)
    b = bits_to_int(b_bits)
    n_bits = half
    
    return a, b, n_bits


def recursive_inference(model, x, y, n_bits, base_case_bits=4, n_loops=8, depth=0):
    """Autoregressively generate Karatsuba trace with recursive self-invocation.
    The model generates tokens one by one. On [PAUSE], we recurse.
    Returns (predicted_product, token_count)."""
    
    # Build input prefix (teacher-forced: we know the input)
    x_bits = int_to_bits(x, n_bits)
    y_bits = int_to_bits(y, n_bits)
    
    token_ids = [INPUT_ID]
    bit_sigs = [200 + STEP_INPUT]
    step_types = [STEP_INPUT]
    
    for i, b in enumerate(x_bits):
        token_ids.append(bit_to_token(b))
        bit_sigs.append(i)
        step_types.append(STEP_INPUT)
    for i, b in enumerate(y_bits):
        token_ids.append(bit_to_token(b))
        bit_sigs.append(i)
        step_types.append(STEP_INPUT)
    
    total_tokens = len(token_ids)
    max_gen = 300  # safety limit
    current_step_type = STEP_INPUT
    current_bit_idx = 0
    
    for _ in range(max_gen):
        # Predict next token
        next_tok = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)
        
        if next_tok == PAUSE_ID:
            # Model wants to recurse!
            # Extract sub-problem from preceding tokens
            sub = extract_sub_problem(token_ids)
            if sub is None:
                return None, total_tokens
            
            a, b, sub_n_bits = sub
            
            # Pad to next power of 2 if needed
            padded = 1
            while padded < sub_n_bits:
                padded <<= 1
            padded = max(padded, 4)
            
            # Recursive call!
            sub_result, sub_tokens = recursive_inference(
                model, a, b, padded, base_case_bits, n_loops, depth + 1
            )
            total_tokens += sub_tokens
            
            if sub_result is None:
                return None, total_tokens
            
            # Append [PAUSE]
            token_ids.append(PAUSE_ID)
            bit_sigs.append(200 + STEP_PAUSE)
            step_types.append(STEP_PAUSE)
            
            # Inject [RESUME] + result bits
            result_n_bits = 2 * sub_n_bits
            result_bits = int_to_bits(sub_result, result_n_bits)
            
            token_ids.append(RESUME_ID)
            bit_sigs.append(200 + STEP_RESUME)
            step_types.append(STEP_RESUME)
            
            for i, b in enumerate(result_bits):
                token_ids.append(bit_to_token(b))
                bit_sigs.append(i)
                step_types.append(STEP_RESUME)
            
            current_step_type = STEP_RESUME
            current_bit_idx = 0
            continue
        
        elif next_tok == OUTPUT_ID:
            # Model says it's done. Append [OUTPUT] then generate product bits.
            token_ids.append(OUTPUT_ID)
            bit_sigs.append(200 + STEP_OUTPUT)
            step_types.append(STEP_OUTPUT)
            
            product_n_bits = 2 * n_bits
            output_bits = []
            for i in range(product_n_bits):
                next_bit = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)
                token_ids.append(next_bit)
                bit_sigs.append(i)
                step_types.append(STEP_OUTPUT)
                output_bits.append(token_to_bit(next_bit))
            
            total_tokens += len(token_ids)
            return bits_to_int(output_bits), total_tokens
        
        else:
            # Normal token — append with position metadata
            if next_tok in ALL_TAG_IDS:
                current_step_type = TAG_TO_STEP.get(next_tok, current_step_type)
                current_bit_idx = 0
                bit_sigs.append(200 + current_step_type)
            else:
                bit_sigs.append(current_bit_idx)
                current_bit_idx += 1
            
            token_ids.append(next_tok)
            step_types.append(current_step_type)
    
    # Hit max_gen without OUTPUT
    return None, total_tokens


# Sanity check: 4-bit base case (autoregressive, no recursion)
print('=== Autoregressive Test: 4-bit base case ===')
correct, total = 0, 0
rng_test = random.Random(123)
for _ in range(50):
    a = rng_test.randint(0, 15)
    b = rng_test.randint(0, 15)
    result, _ = recursive_inference(model, a, b, 4, base_case_bits=4)
    if result == a * b:
        correct += 1
    total += 1
print(f'4-bit autoregressive: {correct}/{total} = {correct/total:.1%}')

# Sanity check: 8-bit with recursion
print('\n=== Autoregressive Test: 8-bit with recursion ===')
correct, total = 0, 0
for _ in range(50):
    a = rng_test.randint(0, 255)
    b = rng_test.randint(0, 255)
    result, _ = recursive_inference(model, a, b, 8, base_case_bits=4)
    if result == a * b:
        correct += 1
    else:
        if total < 3:
            print(f'  Error: {a}*{b}={a*b}, got {result}')
    total += 1
print(f'8-bit autoregressive: {correct}/{total} = {correct/total:.1%}')

In [ ]:
# Cell 6: Full evaluation — 8/16/32/64-bit
import time

print('=' * 60)
print('RECURSIVE SELF-INVOCATION — AUTOREGRESSIVE GENERATION')
print('Model trained on 4-bit + 8-bit single-level traces ONLY')
print('Model decides when to PAUSE; runtime manages call stack')
print('=' * 60)

results = {}
for n_bits in [8, 16, 32, 64]:
    max_val = 1 << n_bits
    n_samples = 200 if n_bits <= 16 else 50 if n_bits <= 32 else 20
    rng_eval = random.Random(999)
    
    correct, total, errors = 0, 0, []
    t0 = time.time()
    
    for _ in range(n_samples):
        a = rng_eval.randint(0, max_val - 1)
        b = rng_eval.randint(0, max_val - 1)
        expected = a * b
        result, n_tok = recursive_inference(model, a, b, n_bits, base_case_bits=4)
        
        if result == expected:
            correct += 1
        else:
            errors.append((a, b, expected, result))
        total += 1
    
    elapsed = time.time() - t0
    acc = correct / total
    results[n_bits] = acc
    
    print(f'\n{n_bits:3d}-bit: {correct}/{total} = {acc:.1%}  ({elapsed:.1f}s)')
    if errors[:3]:
        for a, b, exp, got in errors[:3]:
            print(f'  Error: {a} * {b} = {exp}, got {got}')

print('\n' + '=' * 60)
print('COMPARISON TABLE')
print('=' * 60)
print(f"{'Method':<40} {'8-bit':>8} {'16-bit':>8} {'32-bit':>8} {'64-bit':>8}")
print('-' * 72)
print(f"{'Phase 5: Flat sequence':<40} {'99.0%':>8} {'0.0%':>8} {'N/A':>8} {'N/A':>8}")
print(f"{'Phase 6: Teacher-forced recursive':<40} {'100%':>8} {'100%':>8} {'100%':>8} {'N/A':>8}")
r = results
print(f"{'Phase 7: Autoregressive recursive':<40}", end='')
for nb in [8, 16, 32, 64]:
    if nb in r:
        print(f'{r[nb]:>7.1%} ', end='')
    else:
        print(f'{"N/A":>8}', end='')
print()
print('=' * 60)

In [ ]:
# Cell 7: Save model to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_phase7_pause_resume.eqx'), model)
print(f'Model saved to {CKPT_DIR}')